# 3 Edge detection

At its core, an **edge** represents a significant local change in image appearance, typically in intensity or color. These changes often occur at object boundaries, but they can also be caused by variations in surface orientation, illumination, material, or texture. Edges therefore provide a compact representation of the most relevant structures in an image. The process of identifying these transitions is called **edge detection**. By detecting edges we can, for example:

- **Segment objects and regions:** Edges provide cues about where one region ends and another begins, helping to delineate object boundaries and separate regions in an image.
- **Recognize objects:** The shape and spatial arrangement of edges can provide distinctive features that help identify objects and structures.
- **Recover 3D structure:** Edges and other image features can be matched across different views to infer geometry, depth, and the three-dimensional structure of a scene.



### What's an edge?

As commented, an **edge** can be modeled as a transition between two image regions with different intensity levels.  
A simple way to understand this is through the **1D continuous model of an ideal edge** shown below.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/model_of_ideal_edge.PNG" width="600"></center>

In this model, the edge is characterized by three main parameters:

- **Intensity increment** $(H = A-B)$: the difference in intensity between the bright and dark regions.  
  A larger value of $H$ means a stronger contrast and, in general, an easier edge to detect.

- **Slope angle** $(\alpha)$: describes how abrupt the transition is.  
  A steep transition corresponds to a sharp edge, while a smoother transition produces a more gradual edge, as may happen with blur or shadows.

- **Midpoint** $(x_0)$: the spatial position of the center of the transition, which we will use as the location of the edge.

However, the idealized concept of an edge represented as a continuous model doesn't fully capture the complexity found in **discrete** digital images, which are subject to **noise** and **resolution** limitations.

### Error types related to edge detection

Finding edges properly is not a straightforward task, as there exist different errors that can appear when applying edge detection techniques:
- **Detection error.** A good detector exhibits a low ratio of false negative and false positive, that is:
  - False negatives: Existing edges that are not detected.
  - False positives: Detected objects that are not real.
- **Localization error.** Edges are detected, but they are not at the real, exact position.
- **Multiple response.** Multiple detections are raised for the same edge (the edge is thick).

The following figure illustrates such errors.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/edge_errors.jpg" width="400"></center>

Thereby, when designing a good edge detector, the goal is to achieve low detection and localization errors, as well as to avoid multiple responses.

# 3.1 Operators based on first derivative (gradient)

In the upcoming sections, we are going to investigate and implement different edge detection methods. All of them are based on our dear convolution operator, having their own pros and cons.

Concretely, in this notebook we will cover **first-derivative** based operators, which try to detect borders by looking at abrupt intensity differences in neighbor pixels. In the image below we can see two functions $f(x)$ (first row) and how their derivatives (second row) reach their maximum values at the points where the functions' values change more abruptly (around $x_o$).

<center><br /><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/first_derivative_of_continuous_function.PNG" width="700">$\\[20pt]$</center>

If we are dealing with a **two-dimensional** continuous function $F(x,y)$, its derivative is a *vector* (**gradient**) defined as:$\\[10pt]$
$$
\nabla  F(x,y) =
\begin{bmatrix}
F_x \\
F_y
\end{bmatrix}
=
\begin{bmatrix}
\frac{\partial F(x,y)}{\partial x} \\ \frac{\partial F(x,y)}{\partial y}
\end{bmatrix}
$$

where, for simplicity, we use $F_x$ and $F_y$ to denote the partial derivatives
$F_x(x,y)$ and $F_y(x,y)$.

This vector points at the *direction* of maximum (positive) variation of $F(x,y)$:$\\[10pt]$

$$
\alpha(x,y) = \text{atan2} (F_y,F_x)
$$

and has a *module* (magnitude) proportional to the strength of this variation:$\\[10pt]$

$$
|\nabla  F(x,y)| = \sqrt{F_x^2 + F_y^2} \approx |F_x| + |F_y|
$$

The image below shows examples of gradient vectors:$\\[10pt]$

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/gradient_examples.PNG" width="700">$\\[10pt]$</center>

Concretely, the techniques based on the first derivative explored here are:

- Discrete approximations of a **gradient operator** (Sobel, Prewitt, Roberts, etc., <a href="#311">Section 3.1.1</a>).
- The **Derivative of Gaussian** (DroG) operator (<a href="#312">Section 3.1.2</a>).

## Problem context - Edge detection for medical images

Edge detection in medical images is of capital importance for the diagnosis of different diseases (e.g., the detection of tumor cells) in human organs such as lungs and prostates, becoming an essential pre-processing step in medical image segmentation.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/hospital_clinico.jpg" width="400"><br /><br /></center>

In this context, *Hospital Clínico*, a very busy hospital in Málaga, is asking local engineering students to join their research team. They are looking for a person with knowledge in image processing and, in order to ensure it, they have published 3 medical images: `medical_1.jpg`, `medical_2.jpg` and `medical_3.jpg`. They have asked us to perform accurate edge detection in the three images, as well as to provide an explanation of how it has been made.

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 03. Edge detection'
except ImportError:
    pass

import numpy as np
from scipy import signal
import cv2
import matplotlib.pyplot as plt
import matplotlib
from ipywidgets import interactive, fixed, widgets
matplotlib.rcParams['figure.figsize'] = (15.0, 15.0)

images_path = './images/'

To face this challenge, we are going to use plenty edge detection methods, which will be tested and compared in order to determine the best option.

### **<span style="color:green"><b><i>ASSIGNMENT 1: Taking a look at images</i></b></span>**

First, **display the provided images** to get an idea about what we are dealing with.

*Note: As most medical images does not provide color information, we are going to use border detection in grayscale images.*

*Tip: Different approaches can be followed for edge detection in color images, like converting to YCrCb color space (appendix 2), or detecting edges on each RGB channel.*

In [ ]:
# ASSIGNMENT 1
# Display the provided images in a 1x3 plot to see what are we dealing with
# Write your code here!

# Read the images
medical_1 = cv2.imread(images_path + None, 0)
medical_2 = cv2.imread(images_path + None, 0)
medical_3 = cv2.imread(images_path + None, 0)

# And show them
plt.subplot(131)
plt.imshow(None, None)
plt.title('Medical 1')

plt.subplot(132)
plt.imshow(None, None)
plt.title('Medical 2')

plt.subplot(133)
plt.imshow(None, None)
plt.title('Medical 3')
plt.show()

## 3.1.1 Discrete approximations of a gradient operator <a id=311></a>

The first bunch of methods that we are going to explore carry out a **discrete approximation of a gradient operator** based on the differences between gray (intensity) levels. For example, in order to obtain the derivative in the rows' direction, we could apply:

- Backward difference of pixels along a row:

$$
F_x[i,j]
=
\frac{F[i,j]-F[i,j-1]}{\Delta x}
\qquad
\begin{array}{|c|c|c|}
\hline
0 & 0 & 0\\ \hline
0 & \color{teal}{1} & -1\\ \hline
0 & 0 & 0\\ \hline
\end{array}
$$

- Symmetric difference of pixels along a row:

$$
F_x[i,j]
=
\frac{F[i,j+1]-F[i,j-1]}{2\Delta x}
\qquad
\begin{array}{|c|c|c|}
\hline
0 & 0 & 0\\ \hline
1 & \color{teal}{0} & -1\\ \hline
0 & 0 & 0\\ \hline
\end{array}*\frac{1}{2}
$$

where $\Delta x$ is the spatial sampling step. Since image coordinates are
expressed in pixels, we take $\Delta x=1$ pixel.

These approximations are typically implemented through the convolution of the image with a pair of templates $h_x$ (for rows, computing horizontal derivatives for detecting vertical edges) and $h_y$ (for columns, computing vertical derivatives for detecting horizontal ones), that is:

$$
F_x[i,j] = F[i,j] \otimes h_x[i,j] \leftarrow \text{Vertical edges}
$$
$$
F_y[i,j] = F[i,j] \otimes h_y[i,j] \leftarrow \text{Horizontal edges}
$$

*Recall that, since $\otimes$ is a convolution, kernels must be flipped before being applied.*

Perhaps the most popular operator doing this is such of **Sobel**, although there are many of them that provide acceptable results. These operators use the aforementioned two kernels (typically of size $3\times3$ or $5\times5$) which are convolved with the original image to calculate approximations of the derivatives.

## Common discrete gradient operators

Below are some common discrete operators used to approximate the image gradient.
Recall that, for each operator, $h_x$ estimates the derivative along the horizontal direction and $h_y$ along the vertical direction. Notice that the kernerls have been already flipped.

**Roberts**

$$
h_x=
\begin{array}{|c|c|c|}
\hline
0 & 0 & 0\\ \hline
0 & 0 & 1\\ \hline
0 & -1 & 0\\ \hline
\end{array}
\qquad
h_y=
\begin{array}{|c|c|c|}
\hline
-1 & 0 & 0\\ \hline
0 & 1 & 0\\ \hline
0 & 0 & 0\\ \hline
\end{array}
$$

**Prewitt**

$$
h_x=\frac{1}{3}
\begin{array}{|c|c|c|}
\hline
1 & 0 & -1\\ \hline
1 & 0 & -1\\ \hline
1 & 0 & -1\\ \hline
\end{array}
\qquad
h_y=\frac{1}{3}
\begin{array}{|c|c|c|}
\hline
-1 & -1 & -1\\ \hline
0 & 0 & 0\\ \hline
1 & 1 & 1\\ \hline
\end{array}
$$

**Sobel**

$$
h_x=\frac{1}{4}
\begin{array}{|c|c|c|}
\hline
1 & 0 & -1\\ \hline
2 & 0 & -2\\ \hline
1 & 0 & -1\\ \hline
\end{array}
\qquad
h_y=\frac{1}{4}
\begin{array}{|c|c|c|}
\hline
-1 & -2 & -1\\ \hline
0 & 0 & 0\\ \hline
1 & 2 & 1\\ \hline
\end{array}
$$

**Frei-Chen**

$$
h_x=\frac{1}{2+\sqrt{2}}
\begin{array}{|c|c|c|}
\hline
1 & 0 & -1\\ \hline
\sqrt{2} & 0 & -\sqrt{2}\\ \hline
1 & 0 & -1\\ \hline
\end{array}
\qquad
h_y=\frac{1}{2+\sqrt{2}}
\begin{array}{|c|c|c|}
\hline
-1 & -\sqrt{2} & -1\\ \hline
0 & 0 & 0\\ \hline
1 & \sqrt{2} & 1\\ \hline
\end{array}
$$

**In general**

$$
h_x=\frac{1}{2+k}
\begin{array}{|c|c|c|}
\hline
1 & 0 & -1\\ \hline
k & 0 & -k\\ \hline
1 & 0 & -1\\ \hline
\end{array}
\qquad
h_y=\frac{1}{2+k}
\begin{array}{|c|c|c|}
\hline
-1 & -k & -1\\ \hline
0 & 0 & 0\\ \hline
1 & k & 1\\ \hline
\end{array}
$$

Notice that:
- **Prewitt** corresponds to $k=1$,
- **Sobel** to $k=2$,
- **Frei-Chen** to $k=\sqrt{2}$.

At this point we know how to perform a discrete approximation of a gradient operator through the application of a convolution operation with two different kernels, that is:

$$\nabla  F[x,y] =
\begin{bmatrix}
F \otimes h_x \\ F \otimes h_y
\end{bmatrix}$$

But, how could we use the output of those computations to detect edges? The following figure clarifies that!

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/how_to_detect_edges_with_gradient_image2.png" width="350"></center>

Notice that 80 is just an example of possible threshold value.

**Kernel sizes**

As discussed, kernels can be of different size, and that size directly affects the quality of the detection and the localization (e.g. Sobel $3\times3$ or $5\times5$):
- Small template:
  - more precise localization (good localization).
  - more affected by noise (likely produces false positives).
- Large template:
  - less precise localization.
  - more robust to noise (good detector).
  - higher computational cost ($O(N \times N)$).

### **<span style="color:green"><b><i>ASSIGNMENT 2: Playing with Sobel derivatives</i></b></span>**

Now that we have acquired a basic understanding of these methods, let's complete the following code cell to employ the Sobel kernels ($S_x,S_y$) to compute both derivatives and display them along with the original image (`medical_3.jpg`).

*Notice that the derivative image values can be positive **and negative**, caused by the negative values in the kernel. This implies that the desired depth of the destination image (`ddepth`) has to be at least a signed data type when calling to the `filter2D()` method.*

In [ ]:
# ASSIGNMENT 2
# Read one of the images, compute both kernel derivatives, apply them to the image (separately) and display the original image and the derivatives in a 1x3 plot.
# Write your code here!

# Read the image
image = cv2.imread(images_path + 'medical_3.jpg', 0)

# Define horizontal and vertical kernels
kernel_h = np.array([[None],[None],[None]])*None
kernel_v = np.array([[None],[None],[None]])*None

# Apply convolution
d_horizontal = cv2.filter2D(None,None,None) # Using ddepth=cv2.CV_16S
d_vertical = cv2.filter2D(None,None,None)

# And show them!
plt.subplot(131)
plt.imshow(None, None)
plt.title('Original image')

plt.subplot(132)
plt.imshow(None, None)
plt.title('Horizontal derivative')

plt.subplot(133)
plt.imshow(None, None)
plt.title('Vertical derivative');

<font color='blue'>**Expected output:**  </font>

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/result_sobel_horizontal_vertical_derivative.png">

Once we have computed both derivative images $F_x$ and $F_y$, we can determine the *complete* edge image by computing the image gradient magnitude and then binarizing the result. Recall that the image codifying the gradient magnitude can be computed and approximated as:

$$|\nabla F[x,y]| = \sqrt{(F \otimes h_x)^2 + (F \otimes h_y)^2} \approx |F \otimes h_x| + |F \otimes h_y| = |F_x| + |F_y|$$

### **<span style="color:green"><b><i>ASSIGNMENT 3a: Time to detect edges</i></b></span>**

Complete `edge_detection_chart()` that computes the gradient image of an input one using `kernel_h` and `kernel_v` (kernels for horizontal and vertical derivatives respectively) and **binarize the resultant image** (final edges image) using `threshold`. Then display in a 1x3 plot `image`, the gradient image, and finally, an image with the detected edges! (Only if `verbose` is True).

*Tip: usually it's a good idea to [normalize](https://docs.opencv.org/2.4/modules/core/doc/operations_on_arrays.html#normalize) the gradient image before thresholding.*

*Interesting functions: [`np.absolute()`](https://numpy.org/doc/stable/reference/generated/numpy.absolute.html), [`np.add()`](https://numpy.org/doc/stable/reference/generated/numpy.add.html), [`cv2.threshold()`](https://docs.opencv.org/master/d7/d1b/group__imgproc__misc.html#gae8a4a146d1ca78c626a53577199e9c57)*

In [ ]:
# ASSIGNMENT 3a
# Implement a function that that computes the gradient of an image, taking also as input the convolution kernels.
# It must also binarize the resulting image using a threshold
# Show the input image, the gradient image (normalized) and the binarized edge image in a 1x3 plot (if verbose = True)
def edge_detection_chart(image, kernel_h, kernel_v, threshold, verbose=False):
    """ Computed the gradient of the image, binarizes and display it.

        Args:
            image: Input image
            kernel_h: kernel for horizontal derivative
            kernel_v: kernel for vertical derivative
            threshold: threshold value for binarization
            verbose: Only show images if this is True

        Returns:
            edges: edges binary image
    """
    # Write your code here!

    # Compute derivatives
    d_h = cv2.filter2D(None, None, None) # horizontal
    d_v = cv2.filter2D(None, None, None) # vertical

    # Compute gradient
    gradient_image = None # Hint: You have to sum both derivatives in absolute value!

    #Normalize gradient
    norm_gradient = np.copy(image)
    norm_gradient = cv2.normalize(None, None, None, None, cv2.NORM_MINMAX)

    # Threshold to get edges
    ret, edges = cv2.threshold(None, None, None,cv2.THRESH_BINARY)

    if verbose:
        # Show the initial image
        plt.subplot(131)
        plt.imshow(None, None)
        plt.title('Original image')

        # Show the gradient image
        plt.subplot(132)
        plt.imshow(None, None)
        plt.title('Gradient image')

        # Show edges image
        plt.subplot(133)
        plt.imshow(None, None)
        plt.title('Edges detected')

    return edges

You can use next code to **test if your results are correct**:

In [ ]:
image = np.array([[10,60,20],[60,22,74],[72,132,2]], dtype=np.uint8)

# Sobel derivatives
kernel_h = np.array([[1,0,-1],[2,0,-2],[1,0,-1]])*1/4
kernel_v = np.array([[-1,-2,-1],[0,0,0],[1,2,1]])*1/4

print(edge_detection_chart(image, kernel_h, kernel_v, 100))

<font color='blue'>**Expected output:**  </font>

    [[  0   0   0]
     [255 255 255]
     [  0 255   0]]

### **<span style="color:green"><b><i>ASSIGNMENT 3b: Testing our detector</i></b></span>**

Now **try the implemented method** with different size Sobel kernels ($3\times3$, $5\times5$, ...). $\\[5pt]$  

In [ ]:
# ASSIGNMENT 3b
# Read the image, set you kernels (Sobel, Roberts, Prewitt, etc.) and interact with the threshold parameter.
# Write your code here!

# Read image
image = cv2.imread(images_path + 'medical_3.jpg', 0)

# Define kernel (Sobel)
kernel_h = None
kernel_v = None

#Interact with your code!
interactive( edge_detection_chart, image=fixed(image), kernel_h=fixed(kernel_h), kernel_v=fixed(kernel_v), threshold=(0,255,10), verbose=fixed(True))

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

Try other edge detection operators based on the first derivative with different kernel sizes (Roberts, Prewitt, etc.).

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now, **answer the following questions**:

- What happens if we use a bigger kernel?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- There are differences between Sobel and other operators?
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>  
    
- What errors appear using those operators?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>  
    
- Why kernels usually are divided by a number? (e.g. $3\times3$ Sobel is divided by 4)

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>  

## 3.1.2 DroG operator <a id=312></a>

Despite the simplicity of the previous techniques, they have a remarkable drawback: their performance is highly influenced by image noise. Taking a look at the following figure we can see how, having an apparently not so noisy function (first row), where it is easy to visually detect a step (an abrupt change in its values) around 1000, the response of the derivative with that level of noise is as bigger as the step itself!$\\[5pt]$

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/edge_detection/images/example_of_derivative_of_noisy_function.PNG" width="600"><br /></center>
<center><i>Source: S. Seitz</i></center>$\\[5pt]$

But not everything is lost! An already studied image processing technique can be used to mitigate such noise: **image smoothing**, and more concretely, **Gaussian filtering!** The basic idea is to smooth the image and then apply a gradient operator, that is to compute $\frac{\partial}{\partial x}(f \otimes g)$, being $g$ the Gaussian filter. Not only that, this can be done even more efficiently thanks to the convolution derivative property:$\\[5pt]$

$$\frac{\partial}{\partial x}(f \otimes g) = f \otimes \frac{\partial}{\partial x}g
$$$\\[5pt]$

That is, precomputing the resultant kernels from the convolution of the Gaussian filtering and the Sobel ones, and then convolving them with the image to be processed. With that we save one operation!$\\[5pt]$

This combination of smoothing and gradient is usually called **Derivative of Gaussian operator (DroG)**. Formally:$\\[10pt]$

$$
\nabla [f(x,y)\otimes g_\sigma(x,y)] =
f(x,y) \otimes \nabla [g_\sigma(x,y)] =
f(x,y) \otimes \text{DroG} (x,y)
$$$\\[5pt]$

$$
DroG(x,y) =
\nabla \left[ g_\sigma (x,y) \right] =
\underbrace{\begin{bmatrix}
\frac{\partial}{\partial x}[g_\sigma(x)g_\sigma(y)] \\
\frac{\partial}{\partial y}[g_\sigma(x)g_\sigma(y)] \end{bmatrix}}_{\text{separability}} =
\underbrace{\begin{bmatrix} \frac{-xg_\sigma(x)g_\sigma(y)}{\sigma^2} \\ \frac{-yg_\sigma(x)g_\sigma(y)}{\sigma^2}\end{bmatrix}}_{g(x)'=-xg(x)/\sigma^2} =
\begin{bmatrix} \frac{-xg_\sigma(x,y)}{\sigma^2} \\ \frac{-yg_\sigma(x,y)}{\sigma^2}\end{bmatrix}
$$
$\\[5pt]$

Recall that $g_\sigma(x,y)$ is just the 2D gaussian kernel. We worked with it in Chapter 2!

Also remember from the previous notebooks the expression of the Gaussian distribution with 2 variables centered at the origin of coordinates, where the standard deviation $\sigma$ controls the degree of smoothness:

$$g_{\sigma}(x,y) = \frac{1}{2\pi\sigma^2}exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)$$

Take into account that the DroG template or kernel is **created just once**! Then it can applied to as many images as you want.

### **<span style="color:green"><b><i>ASSIGNMENT 4: Applying DroG</i></b></span>**

We would like to try this robust edge detection technique, so complete the `gaussian_kernel()` method that:

1. constructs a 2D gaussian filter (that is, $g_\sigma(x,y)$ in the previous DroG definition) from a 1D one, and
2. derives it, getting the DroG template (in other words, compute $-x g_\sigma(x,y)/\sigma^2$ and $-y g_\sigma(x,y)/\sigma^2$).
3. Finally, it calls our function `edge_detection_chart()`, but using the DroG template instead of the Sobel one.

Its inputs are:
- an image to be processed,
- the kernel aperture size,
- the standard deviation, and
- the gradient image binarization threshold.

In [ ]:
# ASSIGNMENT 4
# Implement a function that builds the horizontal and vertical DroG templates and calls to 'edge_detection_chart' using that templates.
# Inputs: an image, the kernel aperture size, the Gaussian standard deviation and the threshold for the gradient image binarization
# It returns the horizontal and vertical kernels
def drog_kernel(image, w_kernel, sigma, threshold, verbose=False):
    """ Construct the DroG operator and call edge_detection_chart.

        Args:
            image: Input image
            w_kernel: Kernel aperture size
            sigma: Standard deviation of the Gaussian distribution
            threshold: Threshold value for binarization
            verbose: Only show images if this is True

        Returns:
            DroG_h, DroG_v: DroG kernel for computing horizontal and vertical derivatives
    """
    # Write your code here!

    # Create the 1D gaussian filter
    s = sigma
    w = w_kernel
    gaussian_kernel_1D = np.array([None for z in range(-w,w+1)])

    # Get the 2D gaussian filter from the 1D one.
    vertical_kernel = gaussian_kernel_1D.reshape(2*w+1,1)
    horizontal_kernel = gaussian_kernel_1D.reshape(1,2*w+1)
    gaussian_kernel_2D = signal.convolve2d(None, None)

    # Construct DroG

    # Define x and y axis
    x = np.arange(-w,w+1)
    y = np.vstack(x)

    # Get the kernels for detecting horizontal and vertical edges
    DroG_h = None*(-None)/None**2 # Horizontal derivative
    DroG_v = None*(-None)/None**2 # Vertical derivative

    # Call edge detection chart using DroG
    edge_detection_chart(None, None, None, None, None)

    return DroG_h, DroG_v

You can use next code to **test if results are correct**:

In [ ]:
# Create an input image
image = np.array([[10,60,20],[60,22,74],[72,132,2]], dtype=np.uint8)

# Apply the Gaussian kernel
drog_kernel(image, w_kernel=1, sigma=1.2, threshold=100)

<font color='blue'>**Expected output:**  </font>

    (array([[ 0.03832673, -0.        , -0.03832673],
            [ 0.05423735, -0.        , -0.05423735],
            [ 0.03832673, -0.        , -0.03832673]]),
     array([[ 0.03832673,  0.05423735,  0.03832673],
            [-0.        , -0.        , -0.        ],
            [-0.03832673, -0.05423735, -0.03832673]]))

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now **try this method** and play with its interactive parameters in the next code cell. Then **answer the following questions**:

- What happens if a bigger kernel is used?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What kind of errors appear and disappear whenever sigma is modified?
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   
    
- Why the gradient image have lower values than the one from the original image? *Tip: [image normalization](https://stackoverflow.com/questions/40645985/opencv-python-normalize-image/42164670)*

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   
    
- Now that you have tried different techniques, in your opinion, which is the best one for this type of images?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   

In [ ]:
# Read the image
image = cv2.imread(images_path + 'medical_3.jpg', 0)

# Interact with the three input parameters
interactive(drog_kernel, image=fixed(image), w_kernel=(1,5,1), sigma=(0.4,5,0.5), threshold=(0,255,10), verbose=fixed(True))

## <font color="orange"><b><i>AI Appendix</i></b></font>

### A.1 How I used AI
- **Model/tool (if any):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I asked AI to help with (bullets):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **Best prompt I used (paste):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I kept vs. changed from the AI output:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### A.2 AI review
Ask one generative AI to review your memo using this prompt:

```
Act as a technical reviewer of this computer vision lab notebook. Read my memo and return exactly 5 bullet points:

- two on theory (key ideas),
- two on practice (relevant programming concepts of python used, not particular functions or variables),
- one on how to improve the code (most critical aspect: limitations, parameter choices, runtime/robustness, possible bugs).

Keep each bullet ≤15 words and make them specific to my work.
```

Paste its **5** items here:

- **(Theory 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Theory 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Improvement)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

Finally, **do you agree with them?**, **would you have selected others?**
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## Conclusion

Awesome! Now you have expertise in more applications of the convolution operator. In this notebook you:

- Learned basic operators for edge detection that perform a **discrete approximation of a gradient operator**.
- Learned **how to construct a DroG kernel** in an efficient way.
- Played a bit with them in the context of medical images, discovering some real and meaningful utilities.